# CAREER Exploit Text Graph over HackerSignal

This notebook builds the Exploit Text Graph (ETG) described in the
CAREER proposal using the notebook-local HackerSignal JSONL corpus at
`ETG_MISQ/data/career_rt1_hackersignal.jsonl`. It does **not** run the
HackerSignal benchmark tasks.

The ETG is a temporal graph sequence. For each time spell:

- word nodes represent stemmed exploit/vulnerability terms;
- term-term edges represent directed sliding-window co-occurrence;
- each cumulative graph `G_t` contains words and co-occurrences from
  posts in or before time spell `t`;
- node features include count, document count, degree, weighted degree,
  lexical flags, and trigram-hash features.

The output is the graph itself: per-spell cumulative node and edge
tables, graph statistics, top hubs, observed CVE context summaries, and
temporal drift terms. It then trains the RT1.2 Diachronic Graph
Transformer stage and runs the RT1.3 intrinsic/extrinsic evaluation
benchmark ladder.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd()
if (cwd / "pyproject.toml").exists():
    REPO = cwd
elif (cwd.parent / "pyproject.toml").exists():
    REPO = cwd.parent
else:
    raise RuntimeError(f"Cannot locate repo root from {cwd}")
os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from etg.career_hackersignal_pipeline import (
    build_spell_etgs,
    build_spell_etgs_streaming,
    choose_runtime_plan,
    corpus_profile,
    data_file_fingerprint,
    etg_stats_frame,
    load_cached_pickle,
    load_hackersignal_posts,
    run_dgt_pipeline,
    run_rt13_baseline_evaluation,
    save_cached_pickle,
    streaming_corpus_profile,
    stable_fingerprint,
    temporal_shift_terms,
    write_etg_artifacts,
)
from etg.career_rt1_benchmarks import run_rt1_benchmark_experiments

# Default behavior: run the full CAREER RT1 HackerSignal ETG from
# ETG_MISQ/data/career_rt1_hackersignal.jsonl when CUDA is available,
# or the notebook-local smoke file on MPS/CPU.
#
# Local validation override:
#   ETG_RUN_MODE=smoke
#
# To force a hardware choice:
#   ETG_DEVICE=cuda|mps|cpu
plan = choose_runtime_plan(
    repo_root=REPO,
    requested_device=os.environ.get("ETG_DEVICE", "auto"),
    requested_mode=os.environ.get("ETG_RUN_MODE", "auto"),
)

DATA_PATH = Path(plan.data_path)
FIGURE_DIR = Path("ETG_MISQ/figures")
OUTPUT_DIR = Path("ETG_MISQ/output")
CACHE_DIR = OUTPUT_DIR / "cache"
GRAPH_DIR = OUTPUT_DIR / "etg_graph"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("device:", plan.device)
print("mode:", plan.mode)
print("data:", DATA_PATH, DATA_PATH.exists())
print("record_limit:", plan.record_limit)
print("etg_settings:", plan.__dict__)
print("cache:", CACHE_DIR)


## 2. HackerSignal Corpus Profile


In [ ]:
# Full HackerSignal ETG: use all source layers in the unified corpus.
# Set this to {"hacker_community"} only if you want the narrower RT1
# forum-only interpretation from the proposal.
LAYER_FILTER = None
posts = []

profile_fp = stable_fingerprint({
    "stage": "corpus_profile",
    "data": data_file_fingerprint(DATA_PATH),
    "mode": plan.mode,
    "record_limit": plan.record_limit,
    "layer_filter": LAYER_FILTER,
    "min_date": plan.min_date,
})
profile_cache = CACHE_DIR / "corpus_profile.pkl"
cached_profile = load_cached_pickle(profile_cache, profile_fp)

if cached_profile is not None:
    print(f"[cache] loaded corpus profile from {profile_cache}")
    posts = cached_profile.get("posts", [])
    profile_summary = cached_profile["profile_summary"]
    n_loaded = cached_profile["n_loaded"]
elif plan.mode == "smoke":
    posts = load_hackersignal_posts(DATA_PATH, limit=plan.record_limit, layer_filter=LAYER_FILTER,
                                    min_date=plan.min_date)
    profile = corpus_profile(posts)
    n_loaded = len(posts)
    profile_summary = (
        profile.groupby("source_layer")
        .agg(
            records=("source_layer", "size"),
            sources=("source_dataset", "nunique"),
            forums=("forum_id", "nunique"),
            first_year=("year", "min"),
            last_year=("year", "max"),
            median_tokens=("tokens", "median"),
            cve_hit_rate=("has_cve", "mean"),
        )
        .assign(cve_hit_rate=lambda d: (100 * d["cve_hit_rate"]).round(1))
        .sort_values("records", ascending=False)
    )
    save_cached_pickle(profile_cache, {"posts": posts, "profile_summary": profile_summary, "n_loaded": n_loaded}, profile_fp)
    print(f"[cache] saved corpus profile to {profile_cache}")
else:
    profile = streaming_corpus_profile(DATA_PATH, limit=plan.record_limit, layer_filter=LAYER_FILTER,
                                       min_date=plan.min_date)
    n_loaded = int(profile["records"].sum())
    profile_summary = (
        profile.groupby("source_layer")
        .agg(
            records=("records", "sum"),
            sources=("source_dataset", "nunique"),
            forums=("forum_id", "nunique"),
            first_year=("first_year", "min"),
            last_year=("last_year", "max"),
            mean_tokens=("mean_tokens", "mean"),
            cve_hit_rate=("cve_hit_rate", "mean"),
        )
        .round({"mean_tokens": 1, "cve_hit_rate": 1})
        .sort_values("records", ascending=False)
    )
    save_cached_pickle(profile_cache, {"profile_summary": profile_summary, "n_loaded": n_loaded}, profile_fp)
    print(f"[cache] saved corpus profile to {profile_cache}")

print(f"Loaded {n_loaded:,} dated HackerSignal records for ETG construction.")
display(profile_summary)

## 3. Build the Temporal Exploit Text Graph


In [ ]:
if plan.mode == "full":
    snapshots, vocab = build_spell_etgs_streaming(
        DATA_PATH,
        n_spells=plan.n_spells,
        vocab_size=plan.vocab_size,
        window=plan.window,
        min_edge_weight=plan.min_edge_weight,
        max_edges_per_spell=plan.max_edges_per_spell,
        limit=plan.record_limit,
        layer_filter=LAYER_FILTER,
        min_date=plan.min_date,
        cache_path=CACHE_DIR / "rt11_etg_snapshots.pkl",
        cache_metadata={"mode": plan.mode, "data_path": str(DATA_PATH)},
    )
else:
    snapshots, vocab = build_spell_etgs(
        posts,
        n_spells=plan.n_spells,
        vocab_size=plan.vocab_size,
        window=plan.window,
        min_edge_weight=plan.min_edge_weight,
        max_edges_per_spell=plan.max_edges_per_spell,
        cache_path=CACHE_DIR / "rt11_etg_snapshots.pkl",
        cache_metadata={"mode": plan.mode, "data_path": str(DATA_PATH)},
    )

etg_stats = etg_stats_frame(snapshots)
display(etg_stats)

manifest = write_etg_artifacts(snapshots, GRAPH_DIR)
print(f"wrote ETG graph artifacts to {GRAPH_DIR}")
display(pd.DataFrame(manifest["spells"]))

## 4. ETG Growth Across Time


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
x = range(len(etg_stats))
ax1.bar(x, etg_stats["word_nodes"], color="#4C78A8", alpha=0.78, label="Word nodes")
ax2.plot(x, etg_stats["edges"], marker="o", linewidth=2, color="#F58518", label="Edges")
ax1.set_xticks(list(x), [f"T{s}" for s in etg_stats["spell"]])
ax1.set_ylabel("Word nodes")
ax2.set_ylabel("Edges")
ax1.set_xlabel("Temporal spell")
ax1.set_title("CAREER Exploit Text Graph Growth over HackerSignal")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "career_etg_growth.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. ETG Hubs and CVE Context


In [ ]:
hub_rows = []
for snap in snapshots:
    for term, count in snap["top_terms"][:10]:
        degree = snap["in_degree"].get(term, 0) + snap["out_degree"].get(term, 0)
        hub_rows.append({"spell": snap["spell"], "kind": "top_word", "item": term, "count": count, "degree": degree})
    for (src, dst), weight in snap["top_edges"][:10]:
        hub_rows.append({"spell": snap["spell"], "kind": "top_edge", "item": f"{src} -> {dst}", "count": weight, "degree": ""})
    for cve, count in snap["top_cves"][:10]:
        hub_rows.append({"spell": snap["spell"], "kind": "observed_cve_context", "item": cve, "count": count, "degree": ""})

hubs = pd.DataFrame(hub_rows)
display(hubs)
hubs.to_csv(OUTPUT_DIR / "career_etg_hubs.csv", index=False)


## 6. Temporal Drift Terms


In [ ]:
shifts = temporal_shift_terms(snapshots, top_k=12, min_count=max(5, plan.min_edge_weight))
display(shifts.head(40))
shifts.to_csv(OUTPUT_DIR / "career_etg_temporal_shift_terms.csv", index=False)

if not shifts.empty:
    top = shifts.head(24).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    labels = top["transition"] + " / " + top["word"]
    ax.barh(labels, top["log2_lift"], color="#54A24B")
    ax.set_xlabel("log2 normalized-frequency lift")
    ax.set_title("Largest Adjacent-Spell ETG Term Increases")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "career_etg_temporal_shift_terms.png", dpi=200, bbox_inches="tight")
    plt.show()


## 7. RT1.2 Diachronic Graph Transformer


In [ ]:
DGT_DIR = OUTPUT_DIR / "dgt"
dgt_result = run_dgt_pipeline(
    snapshots,
    max_nodes=plan.dgt_max_nodes,
    hidden_dim=plan.dgt_hidden_dim,
    heads=plan.dgt_heads,
    layers=plan.dgt_layers,
    epochs=plan.dgt_epochs,
    lap_pe_k=plan.lap_pe_k,
    device=plan.device,
    output_dir=DGT_DIR,
    edge_batch=plan.dgt_edge_batch,                  # None → auto-computed from GPU memory
    temporal_loss_weight=plan.temporal_loss_weight,  # 0.1 to fix ablation inversion
    pe_type=plan.pe_type,                            # v5: "laplacian" | "rwpe" | "mose" | "none"
    use_residual_bypass=plan.use_residual_bypass,    # v5: learnable alpha bypass
    temporal_gate=plan.temporal_gate,                # v5: per-node GRU-style gate
    use_time_embedding=plan.use_time_embedding,      # v5: include spell-index embedding
    time_encoding=plan.time_encoding,                # v5: "learned_discrete" | "learned_linear"
    rwpe_attention_bias=plan.rwpe_attention_bias,    # v5: GRIT-style RWPE attention bias
    use_trend_seasonal=plan.use_trend_seasonal,      # v5: TIDFormer trend+seasonal loss
)

print("DGT metrics:")
print(json.dumps(dgt_result["metrics"], indent=2))

display(dgt_result["shifts"].head(40))
display(dgt_result["predictions"].head(40))

## 8. RT1.3 Intrinsic and Extrinsic Evaluation Benchmarks


In [ ]:
# Compatibility baseline summary retained from the initial RT1.3 pass.
rt13_eval = run_rt13_baseline_evaluation(
    snapshots,
    dgt_result,
    OUTPUT_DIR / "rt13_baselines",
    max_nodes=plan.dgt_max_nodes,
    dim=plan.dgt_hidden_dim,
    top_k=50,
)
print("RT1.3 quick baseline metrics:")
print(json.dumps(rt13_eval["metrics"], indent=2))

BENCHMARK_DIR = OUTPUT_DIR / "rt1_benchmarks"
benchmark_result = run_rt1_benchmark_experiments(
    snapshots,
    dgt_result,
    BENCHMARK_DIR,
    max_nodes=plan.dgt_max_nodes,
    dim=plan.dgt_hidden_dim,
    top_k=50,
    lap_pe_k=plan.lap_pe_k,
    device=plan.device,
    dgt_ablation_epochs=max(1, min(3, plan.dgt_epochs // 2)),
)
print("RT1 benchmark leaderboard:")
display(benchmark_result["leaderboard"].head(60))
print("Paired DGT significance tests:")
display(benchmark_result["significance"].head(60))
print("RT1.3 intrinsic/extrinsic DGT evaluation:")
display(benchmark_result["rt13_evaluation"])

rt13_summary = {
    "implemented_rt13_outputs": [
        "non-neural and classic diachronic semantic baselines",
        "static graph embedding and dynamic graph neural baselines",
        "modern graph-transformer and contextual-encoder benchmark slots",
        "DGT component and ETG construction ablations",
        "extrinsic clustering metrics: homogeneity and V-measure",
        "extrinsic classification metrics: accuracy and macro-F1",
        "intrinsic relatedness metric over cybersecurity term groups",
        "predictive metrics: MAE, RMSE, MAPE, R-squared, MSLE, and quantile loss",
        "paired DGT-vs-baseline error deltas with bootstrap confidence intervals, sign-flip p-values, and FDR q-values",
    ],
    "benchmark_dir": str(BENCHMARK_DIR),
    "note": "RT1.3 is implemented as evaluation of the proposed DGT embeddings and prediction layer, consistent with the CAREER task text.",
}
(OUTPUT_DIR / "rt13_summary.json").write_text(json.dumps(rt13_summary, indent=2), encoding="utf-8")
print(json.dumps(rt13_summary, indent=2))


## 9. Save RT1 Run Summary


In [ ]:
summary = {
    "data": {
        "data_path": str(DATA_PATH),
        "records_loaded": n_loaded,
        "source_layers": profile_summary["records"].to_dict(),
        "runtime_plan": plan.__dict__,
    },
    "etg": {
        "n_spells": len(snapshots),
        "vocab_size": len(vocab),
        "stats": etg_stats.to_dict(orient="records"),
        "graph_manifest": manifest,
    },
    "dgt": dgt_result["metrics"],
    "rt13": {
        "summary": str(OUTPUT_DIR / "rt13_summary.json"),
        "baseline_metrics": rt13_eval["metrics"],
        "benchmark_dir": str(OUTPUT_DIR / "rt1_benchmarks"),
        "benchmark_manifest": benchmark_result["manifest"],
        "significance_tests": str(OUTPUT_DIR / "rt1_benchmarks" / "rt1_benchmark_dgt_significance.csv"),
    },
}
out = OUTPUT_DIR / "career_rt1_hackersignal_summary.json"
out.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"wrote {out}")


In [ ]:
# ── External Validation 1: CVE Term Emergence (Issues 1 & 3) ──────────────
import json, pickle, numpy as np
from pathlib import Path
from etg.career_cve_validation import fetch_kev, run_cve_emergence_validation

_snap_path = Path("ETG_MISQ/output/cache/rt11_etg_snapshots.pkl")
_emb_path  = Path("ETG_MISQ/output/dgt/dgt_embeddings.npz")
_kev_cache = Path("ETG_MISQ/output/cve_validation/kev_catalog.json")
_out_dir   = Path("ETG_MISQ/output/cve_validation")
_out_dir.mkdir(parents=True, exist_ok=True)

with open(_snap_path, "rb") as _f:
    _snaps, _ = pickle.load(_f)

_emb_dict = dict(np.load(_emb_path, allow_pickle=True))
_kev      = fetch_kev(cache_path=_kev_cache)

cve_val_results = run_cve_emergence_validation(
    emb_dict   = _emb_dict,
    snapshots  = _snaps,
    kev_entries= _kev,
    top_k      = 50,
)

(_out_dir / "cve_validation_results.json").write_text(
    json.dumps(cve_val_results, indent=2), encoding="utf-8"
)

print(f"Transitions evaluated : {cve_val_results['n_transitions_evaluated']}")
print(f"Mean Precision@{cve_val_results['top_k']} : {cve_val_results['mean_precision_at_k']:.3f}")
print(f"Mean AUC-ROC          : {cve_val_results['mean_auc_roc']:.3f}")
print(f"Mean Spearman ρ       : {cve_val_results['mean_spearman_rho']:.3f}")
print()
print("Per-transition breakdown:")
for t in cve_val_results["per_transition"]:
    print(f"  {t['transition']}: KEV={t['n_kev_entries']:3d} "
          f"| vocab_overlap={t['n_cve_terms_in_vocab']:3d} "
          f"| P@{t['top_k']}={t['precision_at_k']:.3f} "
          f"| AUC={t['auc_roc']:.3f} | ρ={t['spearman_rho']:.3f}")

## 10. Design Science Case Study: Semantic Shifts and Forecasts


In [ ]:
# ── Case Study: Semantic Shift Trajectories and Forecasts ──────────────────
import json, pickle, numpy as np, pandas as pd
from pathlib import Path
from etg.career_case_study import run_case_study

_snap_path  = Path("ETG_MISQ/output/cache/rt11_etg_snapshots.pkl")
_emb_path   = Path("ETG_MISQ/output/dgt/dgt_embeddings.npz")
_preds_path = Path("ETG_MISQ/output/dgt/dgt_shift_predictions.csv")
_out_dir    = Path("ETG_MISQ/output/case_study")
_out_dir.mkdir(parents=True, exist_ok=True)

_emb_dict   = dict(np.load(_emb_path, allow_pickle=True))
_preds_df   = pd.read_csv(_preds_path)

cs_results = run_case_study(
    emb_dict        = _emb_dict,
    predictions_df  = _preds_df,
    output_dir      = _out_dir,
    n_spells        = 12,
    watchlist_top_n = 15,
    annotated_only  = True,
    figure          = True,
)

print("=" * 72)
print("SHIFT TRAJECTORIES — 6 CURATED TERMS")
print("=" * 72)
for t in cs_results["shift_trajectories"]:
    print(f"  {t['display'].upper():<12}  peak {t['peak_spell_transition']} "
          f"({t['peak_cosine_shift']:.3f})  spark: {t['sparkline']}")
    print(f"    Event  : {t['peak_event']}")
    print()

print("=" * 72)
print("WATCHLIST — TOP PREDICTED NEXT-SPELL SHIFTS")
print("=" * 72)
for w in cs_results["watchlist"]:
    print(f"  {w['term']:<22}  pred={w['predicted_shift']:.3f}  "
          f"recent={w['recent_shift']:.3f}")
    print(f"    {w['interpretation']}")
    print()

if cs_results.get("figure_path"):
    from IPython.display import Image, display as ipy_display
    ipy_display(Image(cs_results["figure_path"], width=850))


## 11. Walk-Forward Validation (MISQ Reviewer Response)

Addresses the reviewer concern that the main benchmark rests on a single held-out spell.
For each fold k ∈ {7, 8, 9, 10, 11} we train DGT on spells 1..k, run fast baselines
on the same slice, and evaluate EMA shift prediction against the held-out last transition.
Outputs: per-fold MAE/RMSE/ρ, mean ± std aggregate, one-tailed paired t-test vs DGT.

In [ ]:
# ── Walk-Forward Validation ─────────────────────────────────────────────────
import pickle
import sys
from pathlib import Path

# Add ETG_MISQ/ to path so we can import the validation module directly
_etg_misq_dir = REPO / "ETG_MISQ"
if str(_etg_misq_dir) not in sys.path:
    sys.path.insert(0, str(_etg_misq_dir))

from run_walk_forward_validation import run_walk_forward, aggregate_results

WF_DIR = OUTPUT_DIR / "walk_forward"
WF_DIR.mkdir(parents=True, exist_ok=True)

# Load cached snapshots (already built in Section 3)
_snap_cache = CACHE_DIR / "rt11_etg_snapshots.pkl"
if _snap_cache.exists():
    with open(_snap_cache, "rb") as _f:
        _obj = pickle.load(_f)
    _all_snapshots = _obj[0] if isinstance(_obj, tuple) else _obj
else:
    # Fallback: use the snapshots already in memory from Section 3
    _all_snapshots = snapshots

print(f"Loaded {len(_all_snapshots)} spells for walk-forward validation")

# Run walk-forward — uses cached DGT fold outputs on re-run
wf_df = run_walk_forward(
    _all_snapshots,
    WF_DIR,
    folds=[7, 8, 9, 10, 11],
    max_nodes=plan.dgt_max_nodes,   # match main evaluation (6000)
    dgt_epochs=100,                  # reduced from 200; sufficient for walk-forward
    dim=64,
    lap_pe_k=plan.lap_pe_k,
    device=plan.device,
    hidden_dim=plan.dgt_hidden_dim,
    heads=plan.dgt_heads,
    layers=plan.dgt_layers,
)

wf_agg = aggregate_results(wf_df, WF_DIR)

print()
print("=" * 64)
print("WALK-FORWARD: per-fold MAE")
print("=" * 64)
display(
    wf_df.pivot(index="fold", columns="model", values="mae")
        .round(4)
        .sort_index()
)

print()
print("=" * 64)
print("WALK-FORWARD: aggregate MAE (mean ± std across 5 folds)")
print("=" * 64)
display(
    wf_agg[["mae_mean", "mae_std", "spearman_rho_mean", "spearman_rho_std"]]
        .sort_values("mae_mean")
)

print()
print("=" * 64)
print("WALK-FORWARD: significance (one-tailed paired t-test vs DGT)")
print("=" * 64)
import pandas as pd
wf_sig = pd.read_csv(WF_DIR / "wf_significance.csv")
display(
    wf_sig[["model", "dgt_mean_mae", "mean_mae", "delta_vs_dgt", "ttest_p", "sig_dgt_better"]]
        .sort_values("delta_vs_dgt", ascending=False)
)

# Summary line suitable for the paper
_dgt_rows = wf_df[wf_df["model"] == "dgt"]
print()
print(f"DGT walk-forward MAE : {_dgt_rows['mae'].mean():.4f} "
      f"± {_dgt_rows['mae'].std():.4f}  (across {len(_dgt_rows)} folds)")
print(f"DGT walk-forward ρ   : {_dgt_rows['spearman_rho'].mean():.3f} "
      f"± {_dgt_rows['spearman_rho'].std():.3f}")
